# Week 2 — Inside an LLM
**ESE · AI for Business and FinTech · 30 September 2026**

The sections follow the slides. From section 3 you run a real language model, GPT-2 small (124 million weights), in this notebook.

In [ ]:
!pip -q install transformers tiktoken

## 1. Your tiny LLM, piece 1: a tokenizer

Find the pair of symbols that appears most often; give it a new name; repeat. This is byte-pair encoding, the method real tokenizers use.

In [ ]:
from collections import Counter

def learn_merges(text, merges=8, show=True):
    seq = [c if c != " " else "_" for c in text]          # _ marks a space
    for step in range(1, merges + 1):
        pairs = Counter(zip(seq, seq[1:]))
        (a, b), n = max(pairs.items(), key=lambda kv: (kv[1], -list(pairs).index(kv[0])))
        if n < 2:
            break
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == (a, b):
                out.append(a + b); i += 2
            else:
                out.append(seq[i]); i += 1
        seq = out
        if show:
            print(f"merge {step}: {a!r} + {b!r} seen {n} times -> {a+b!r:10}  text is now {len(seq)} symbols")
    return seq

text = ("the bank left rates unchanged. the bank said rates will stay unchanged. "
        "rates are high and the bank expects rates to stay high.")
print(len(text), "characters to start")
pieces = learn_merges(text)

**Your turn.** A paragraph of your own in English, then the same in Russian. 30 merges each. Symbols per word?

In [ ]:
mine_en = "paste an English paragraph here"
mine_ru = "вставьте сюда абзац на русском"
for name, t in [("English", mine_en), ("Russian", mine_ru)]:
    seq = learn_merges(t, merges=30, show=False)
    print(f"{name}: {len(seq) / len(t.split()):.1f} symbols per word")

## 2. Attention, by hand

"The bank raised rates because **it**" — `it` asks a question (query); every token shows a label (key) and carries a content (value). Two numbers per vector, so you can check every step.

In [ ]:
import numpy as np, pandas as pd

tokens = ["The", "bank", "raised", "rates", "because", "it"]
query  = np.array([1.5, 0.0])                                    # "it": who is an actor that can fear?
keys   = np.array([[0, .2], [2, 0], [0, 1], [1, .5], [0, .5], [.2, 0]])
values = np.array([[0, 0], [1, 0], [0, 0], [0, 1], [0, 0], [0, 0]])   # (is an institution, is a price)

scores  = keys @ query                       # multiply and add: a neuron
weights = np.exp(scores) / np.exp(scores).sum()   # the knobs
print(pd.DataFrame({"score": scores, "e^score": np.exp(scores).round(2), "attention": weights.round(3)}, index=tokens))
print("\nnew meaning of 'it' =", (weights @ values).round(2), " (institution, price)")

**Try:** give `rates` the key `(2, 0)` too. Where does `it` look now — and what would a real model need to tell them apart?

## 3. A real model's knobs

GPT-2 small: 124 million weights, 50,257 tokens, 12 blocks. The first run downloads it (about half a gigabyte).

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
tok = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager")
model.train(False)                                   # inference mode
print(f"{sum(p.numel() for p in model.parameters()):,} weights")

def knobs(prompt, k=8):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        p = torch.softmax(model(ids).logits[0, -1], -1)
    top = torch.topk(p, k)
    print(prompt, "->")
    for v, i in zip(top.values, top.indices):
        print(f"   {tok.decode([int(i)])!r:14} {float(v):6.1%}")

knobs("The ECB left rates")

**Bet first, then run:** three prompts of your own — one from your work, one about markets, one about crypto. How sure is the model about the next word?

In [ ]:
knobs("Bitcoin fell sharply after")

## 4. Temperature, and one token at a time

In [ ]:
prompt = "The ECB left rates"
ids = tok(prompt, return_tensors="pt").input_ids

out = model.generate(ids, max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
print("T = 0  :", prompt + tok.decode(out[0][ids.shape[1]:]))

for T in [1.0, 1.0, 1.0, 2.0]:
    out = model.generate(ids, max_new_tokens=12, do_sample=True, temperature=T, top_k=0, pad_token_id=tok.eos_token_id)
    print(f"T = {T}:", prompt + tok.decode(out[0][ids.shape[1]:]))

**🔍 CHECK.** Which of these sentences contain a number? Is any of those numbers a fact? How would you find out?

## 5. Meaning as a place

Every token is a row of 768 numbers. Cosine similarity: 1 means the same direction, 0 unrelated.

In [ ]:
E = model.transformer.wte.weight.detach()
def similarity(a, b):
    va, vb = E[tok.encode(" " + a)[0]], E[tok.encode(" " + b)[0]]
    return float(torch.nn.functional.cosine_similarity(va, vb, dim=0))

for a, b in [("bank", "lender"), ("bank", "Paris"), ("Bitcoin", "crypto"), ("euro", "dollar")]:
    print(f"{a:8} – {b:8} {similarity(a, b):.2f}")

## 6. Two piles

Ask your assistant: *"Summarise the last annual results of [a company you know]: revenue, profit, employees, the CEO's name."* Split every sentence of the answer:

| Verifiable — a number, date, name or source you can open | Plausible — fluent, and nothing to check it against |
|---|---|
| … | … |

Check two items from the left pile against the company's own report.

### Before you leave: three lines, in your words

1. …
2. …
3. …

Then `File ▸ Save a copy in Drive`, into your `ese-ai-fintech` folder.